# Eva Data

This notebook builds a `MultiDataset` containing exactly one `ZarrDataset`, loads one batch, visualizes one image with `mediapy`, and prints the rest of the batch.

In [ ]:
from pathlib import Path
import torch
import mediapy as mpy
import imageio_ffmpeg
import cv2
from scipy.spatial.transform import Rotation as R

from egomimic.rldb.zarr.zarr_dataset_multi import ZarrDataset, MultiDataset
from egomimic.utils.egomimicUtils import EXTRINSICS
from egomimic.rldb.zarr.action_chunk_transforms import (
    _matrix_to_xyzwxyz,
    build_aria_bimanual_transform_list,
    build_eva_bimanual_transform_list,
)

import numpy as np
from egomimic.utils.egomimicUtils import INTRINSICS, draw_actions, cam_frame_to_cam_pixels, nds

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
# Point this at a single episode directory, e.g. /path/to/episode_hash.zarr
EPISODE_PATH = Path("/coc/flash7/scratch/egoverseDebugDatasets/1767495035712.zarr")

key_map = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "images.right_wrist": {"zarr_key": "images.right_wrist"},
    "images.left_wrist": {"zarr_key": "images.left_wrist"},
    "right.obs_ee_pose": {"zarr_key": "right.obs_ee_pose"},
    "right.obs_gripper": {"zarr_key": "right.gripper"},
    "left.obs_ee_pose": {"zarr_key": "left.obs_ee_pose"},
    "left.obs_gripper": {"zarr_key": "left.gripper"},
    "right.gripper": {"zarr_key": "right.gripper", "horizon": 45},
    "left.gripper": {"zarr_key": "left.gripper", "horizon": 45},
    "right.cmd_ee_pose": {"zarr_key": "right.cmd_ee_pose", "horizon": 45},
    "left.cmd_ee_pose": {"zarr_key": "left.cmd_ee_pose", "horizon": 45},
}

ACTION_CHUNK_LENGTH = 100
ACTION_STRIDE = 1  # set to 3 for Aria-style anchor sampling

extrinsics = EXTRINSICS["x5Dec13_2"]
left_extrinsics_pose = _matrix_to_xyzwxyz(extrinsics["left"][None, :])[0]
right_extrinsics_pose = _matrix_to_xyzwxyz(extrinsics["right"][None, :])[0]

transform_list = build_eva_bimanual_transform_list(
    chunk_length=ACTION_CHUNK_LENGTH,
    stride=ACTION_STRIDE,
    # left_extra_batch_key={"left_extrinsics_pose": left_extrinsics_pose},
    # right_extra_batch_key={"right_extrinsics_pose": right_extrinsics_pose},
)


In [ ]:
# Build a MultiDataset with exactly one ZarrDataset inside
single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map, transform_list=transform_list)
# single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map)
multi_ds = MultiDataset(datasets={"single_episode": single_ds}, mode="total")

print("len(single_ds):", len(single_ds))
print("len(multi_ds):", len(multi_ds))

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)


In [ ]:
batch = next(iter(loader))
nds(batch)

In [ ]:
def _split_action_pose(actions):
    # 14D layout: [L xyz ypr g, R xyz ypr g]
    # 12D layout: [L xyz ypr, R xyz ypr]
    if actions.shape[-1] == 14:
        left_xyz = actions[:, :3]
        left_ypr = actions[:, 3:6]
        right_xyz = actions[:, 7:10]
        right_ypr = actions[:, 10:13]
    elif actions.shape[-1] == 12:
        left_xyz = actions[:, :3]
        left_ypr = actions[:, 3:6]
        right_xyz = actions[:, 6:9]
        right_ypr = actions[:, 9:12]
    else:
        raise ValueError(f"Unsupported action dim {actions.shape[-1]}")
    return left_xyz, left_ypr, right_xyz, right_ypr


def viz_batch(batch, image_key, action_key, intrinsics_key):
    # Image: (C,H,W) -> (H,W,C)
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()

    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS[intrinsics_key]
    actions = batch[action_key][0].detach().cpu().numpy()
    left_xyz, _, right_xyz, _ = _split_action_pose(actions)

    vis = draw_actions(
        img_np.copy(), type="xyz", color="Blues",
        actions=left_xyz, extrinsics=None, intrinsics=intrinsics, arm="left"
    )
    vis = draw_actions(
        vis, type="xyz", color="Reds",
        actions=right_xyz, extrinsics=None, intrinsics=intrinsics, arm="right"
    )
    return vis

In [ ]:
def viz_batch_ypr(batch, image_key, action_key, intrinsics_key, axis_len_m=0.04):
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()

    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS[intrinsics_key]
    actions = batch[action_key][0].detach().cpu().numpy()
    left_xyz, left_ypr, right_xyz, right_ypr = _split_action_pose(actions)

    vis = img_np.copy()

    def _draw_axis_color_legend(frame):
        h, w = frame.shape[:2]
        x_right = w - 12
        y_start = 14
        y_step = 12
        line_len = 24
        axis_legend = [
            ("x", (255, 0, 0)),
            ("y", (0, 255, 0)),
            ("z", (0, 0, 255)),
        ]
        for i, (name, color) in enumerate(axis_legend):
            y = y_start + i * y_step
            x0 = x_right - line_len
            x1 = x_right
            cv2.line(frame, (x0, y), (x1, y), color, 3)
            cv2.putText(
                frame, name, (x0 - 12, y + 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA,
            )
        return frame

    def _draw_rotation_at_palm(frame, xyz_seq, ypr_seq, label, anchor_color):
        if len(xyz_seq) == 0 or len(ypr_seq) == 0:
            return frame

        palm_xyz = xyz_seq[0]
        palm_ypr = ypr_seq[0]
        rot = R.from_euler("ZYX", palm_ypr, degrees=False).as_matrix()
        # print(np.linalg.det(rot))

        axis_points_cam = np.vstack([
            palm_xyz,
            palm_xyz + rot[:, 0] * axis_len_m,
            palm_xyz + rot[:, 1] * axis_len_m,
            palm_xyz + rot[:, 2] * axis_len_m,
        ])

        px = cam_frame_to_cam_pixels(axis_points_cam, intrinsics)[:, :2]
        if not np.isfinite(px).all():
            return frame

        pts = np.round(px).astype(np.int32)

        h, w = frame.shape[:2]
        x0, y0 = pts[0]
        if not (0 <= x0 < w and 0 <= y0 < h):
            return frame

        cv2.circle(frame, (x0, y0), 4, anchor_color, -1)
        axis_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]  # x,y,z as RGB

        for i, color in enumerate(axis_colors, start=1):
            x1, y1 = pts[i]
            if 0 <= x1 < w and 0 <= y1 < h:
                cv2.line(frame, (x0, y0), (x1, y1), color, 2)
                cv2.circle(frame, (x1, y1), 2, color, -1)

        cv2.putText(
            frame, label, (x0 + 6, max(12, y0 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, anchor_color, 1, cv2.LINE_AA,
        )
        return frame

    vis = _draw_rotation_at_palm(vis, left_xyz, left_ypr, "L rot", (255, 180, 80))
    vis = _draw_rotation_at_palm(vis, right_xyz, right_ypr, "R rot", (80, 180, 255))
    vis = _draw_axis_color_legend(vis)
    return vis

In [ ]:
# Separate YPR visualization preview
image_key = "images.front_1"
action_key = "actions_cartesian"

for batch in loader:
    vis_ypr = viz_batch_ypr(batch, image_key=image_key, action_key=action_key, intrinsics_key="base")
    mpy.show_image(vis_ypr)
    break

In [ ]:
image_key = "images.front_1"
action_key = "actions_cartesian"

images = []
for i, batch in enumerate(loader):
    vis = viz_batch(batch, image_key=image_key, action_key=action_key, intrinsics_key="base")
    images.append(vis)
    if i > 100:
        break

mpy.show_video(images, fps=30)

## Human Datasets
Mecka, Scale and Aria should all run exactly the same

In [ ]:
from egomimic.utils.aws.aws_sql import timestamp_ms_to_episode_hash
timestamp_ms_to_episode_hash(1764285211791)

In [ ]:
# Aria-style chunking example: horizon=30 contiguous frames, sample anchors every 3 -> 10 points, then interpolate to 100.

# EPISODE_PATH = Path("/coc/flash7/scratch/egoverseDebugDatasets/scale/697a9070da7b91acaf3f2d88_episode_000000.zarr") # Scale
# intrinsics_key = "scale"

EPISODE_PATH = Path("/coc/flash7/scratch/egoverseDebugDatasets/1767671007927.zarr/") # Aria
intrinsics_key = "base"


key_map = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "right.obs_ee_pose": {"zarr_key": "right.obs_ee_pose"},
    "left.obs_ee_pose": {"zarr_key": "left.obs_ee_pose"},
    "right.action_ee_pose": {"zarr_key": "right.obs_ee_pose", "horizon": 30},
    "left.action_ee_pose": {"zarr_key": "left.obs_ee_pose", "horizon": 30},
    "obs_head_pose": {"zarr_key": "obs_head_pose"},
}

ACTION_CHUNK_LENGTH = 100
ACTION_STRIDE = 3

transform_list = build_aria_bimanual_transform_list(
    chunk_length=ACTION_CHUNK_LENGTH,
    stride=ACTION_STRIDE,
)


In [ ]:
# Build a MultiDataset with exactly one ZarrDataset inside
single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map, transform_list=transform_list)
#single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map)
multi_ds = MultiDataset(datasets={"single_episode": single_ds}, mode="total")

print("len(single_ds):", len(single_ds))
print("len(multi_ds):", len(multi_ds))

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)
# batch = next(iter(loader))

# print("Batch keys:", list(batch.keys()))


In [ ]:
batch = next(iter(loader))
nds(batch)

In [ ]:
image_key = "images.front_1"
action_key = "actions_cartesian"

ims = []
for i, batch in enumerate(loader):
    first_img = batch[image_key][0].detach().cpu().permute(1, 2, 0).numpy()
    first_img = (first_img * 255.0).clip(0, 255).astype(np.uint8)

    vis = viz_batch(batch, image_key=image_key, action_key=action_key, intrinsics_key=intrinsics_key)
    ims.append(vis)
    # mpy.show_image(vis)

    # for k, v in batch.items():
    #     print(f"{k}: {tuple(v.shape)}")
    
    if i > 200:
        break

mpy.show_video(ims, fps=30)


In [ ]:
# Aria YPR video (same data loop, YPR overlay)
image_key = "images.front_1"
action_key = "actions_cartesian"

ims_ypr = []
for i, batch in enumerate(loader):
    vis_ypr = viz_batch_ypr(batch, image_key=image_key, action_key=action_key, intrinsics_key="base")
    ims_ypr.append(vis_ypr)
    if i > 200:
        break

mpy.show_video(ims_ypr, fps=30)

In [ ]:
batch["actions_cartesian"][0, 0]


## Keypoint Visualization

In [ ]:
# Load Scale episode with raw keypoints (no action chunking needed)
from egomimic.rldb.zarr.action_chunk_transforms import _xyzwxyz_to_matrix
from egomimic.utils.egomimicUtils import cam_frame_to_cam_pixels, draw_dot_on_frame
import cv2
import matplotlib.pyplot as plt

EPISODE_PATH_KP = EPISODE_PATH

key_map_kp = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "left.obs_keypoints": {"zarr_key": "left.obs_keypoints"},
    "right.obs_keypoints": {"zarr_key": "right.obs_keypoints"},
    "obs_head_pose": {"zarr_key": "obs_head_pose"},
}

single_ds_kp = ZarrDataset(Episode_path=EPISODE_PATH_KP, key_map=key_map_kp)
multi_ds_kp = MultiDataset(datasets={"single_episode": single_ds_kp}, mode="total")
loader_kp = torch.utils.data.DataLoader(multi_ds_kp, batch_size=1, shuffle=False)
print(f"Keypoint dataset: {len(single_ds_kp)} frames")

In [ ]:
# MANO skeleton edges: (parent, child) for drawing bones
MANO_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # index
    (0, 9), (9, 10), (10, 11), (11, 12),    # middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # ring
    (0, 17), (17, 18), (18, 19), (19, 20),  # pinky
]

FINGER_COLORS = {
    "thumb": (255, 100, 100),   # red
    "index": (100, 255, 100),   # green
    "middle": (100, 100, 255),  # blue
    "ring": (255, 255, 100),    # yellow
    "pinky": (255, 100, 255),   # magenta
}
FINGER_EDGE_RANGES = [
    ("thumb", 0, 4), ("index", 4, 8), ("middle", 8, 12),
    ("ring", 12, 16), ("pinky", 16, 20),
]


def viz_keypoints(batch, image_key="images.front_1"):
    """Visualize all 21 MANO keypoints per hand, projected onto the image."""
    # Prepare image
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()
    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS["scale"]
    head_pose = batch["obs_head_pose"][0].detach().cpu().numpy()  # (6,)

    # T_head_world: camera pose in world (camera-to-world)
    # We need world-to-camera = inv(T_head_world)
    T_head_world = _xyzwxyz_to_matrix(head_pose[None, :])[0]  # (4, 4)
    T_world_to_cam = np.linalg.inv(T_head_world)

    vis = img_np.copy()
    h, w = vis.shape[:2]

    for hand, dot_color in [("left", (0, 120, 255)), ("right", (255, 80, 0))]:
        kps_key = f"{hand}.obs_keypoints"
        if kps_key not in batch:
            continue
        kps_flat = batch[kps_key][0].detach().cpu().numpy()  # (63,)
        kps_world = kps_flat.reshape(21, 3)

        # Skip if keypoints are all zero (invalid, clamped from 1e9)
        if np.allclose(kps_world, 0.0, atol=1e-3):
            continue

        # World -> camera frame
        kps_h = np.concatenate([kps_world, np.ones((21, 1))], axis=1)  # (21, 4)
        kps_cam = (T_world_to_cam @ kps_h.T).T[:, :3]  # (21, 3)

        # Camera frame -> pixels
        kps_px = cam_frame_to_cam_pixels(kps_cam, intrinsics)  # (21, 3+)

        # Identify valid keypoints (z > 0 and in image bounds)
        valid = (kps_cam[:, 2] > 0.01)
        valid &= (kps_px[:, 0] >= 0) & (kps_px[:, 0] < w)
        valid &= (kps_px[:, 1] >= 0) & (kps_px[:, 1] < h)

        # Draw skeleton edges (colored by finger)
        for finger, start, end in FINGER_EDGE_RANGES:
            color = FINGER_COLORS[finger]
            for edge_idx in range(start, end):
                i, j = MANO_EDGES[edge_idx]
                if valid[i] and valid[j]:
                    p1 = (int(kps_px[i, 0]), int(kps_px[i, 1]))
                    p2 = (int(kps_px[j, 0]), int(kps_px[j, 1]))
                    cv2.line(vis, p1, p2, color, 2)

        # Draw keypoint dots on top
        for k in range(21):
            if valid[k]:
                center = (int(kps_px[k, 0]), int(kps_px[k, 1]))
                cv2.circle(vis, center, 4, dot_color, -1)
                cv2.circle(vis, center, 4, (255, 255, 255), 1)  # white border

        # Label wrist
        if valid[0]:
            wrist_px = (int(kps_px[0, 0]) + 6, int(kps_px[0, 1]) - 6)
            cv2.putText(vis, f"{hand[0].upper()}", wrist_px,
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, dot_color, 2)

    return vis

In [ ]:
# Render keypoint video
ims_kp = []
for i, batch_kp in enumerate(loader_kp):
    vis = viz_keypoints(batch_kp)
    ims_kp.append(vis)
    if i > 200:
        break

mpy.show_video(ims_kp, fps=30)